# An End-to-End AI Harvest Planner for Low-Cost Fruit-Picking Robots Built on a Physics-Consistent World Model

**`07_station_sweep.ipynb`**

How many stops should the base make on one tree?

Three was the original answer, chosen because three maximises premium fruit per hour. That is the
right objective for a machine working an unbounded orchard: skim each tree for what is easy to
reach and drive on. It is the wrong objective for this client, who has a finite block and has to
get all of it picked — fruit the robot skips is not saved, it is handed to a person.

The reach diagnostic makes the size of the choice plain. The candidate grid reaches about 63
fruit on an average tree; three stops reach a fifth of them. **Most of what the arm could touch
sits outside the plan, and that is a consequence of the stop count, not of the arm.**

Arithmetic says this should be cheap to fix. Adding a stop costs a few seconds of travel; a pick
costs fourteen. If the extra stops buy picks, they pay for themselves in fruit per second even
though the tree takes longer.

**What this notebook does.** Plans every held-out tree at several stop counts and reports what
each buys and what it costs — under both objectives, so the disagreement between them is on the
page rather than assumed.

Planning is `src/planner.py`; this notebook only varies the stop count and aggregates. Coverage
search and the rate rule are used rather than the trained planner and pick policy, because both
were trained at one setting and would be extrapolating at the others — comparing stop counts
needs everything except the stop count held still. The sweep stage is off throughout for the same
reason: it exists to finish what a stop count leaves behind, and here the stop count is the thing
being measured.


In [1]:
import os, sys, time, json
from pathlib import Path

import numpy as np
import pandas as pd

# Default to the folder this notebook sits in, so a clone runs without setup. An absolute
# default only ever pointed at one machine, and an environment variable set in a shell does
# not reach a kernel that was already running.
ROOT = Path(os.environ.get("AIPICK_ROOT") or Path.cwd())
os.environ["AIPICK_ROOT"] = str(ROOT)
SRC, DATA, MODELS = ROOT/"src", ROOT/"data", ROOT/"models"
OUT = ROOT/"runs"/"stations"; OUT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(SRC))

import environment as E
E.load(ROOT, trees="trees_measured_pose.csv", dynamics=True)

import planner as PL
info = PL.load(ROOT)

TREES = list(range(20, 50))
K_LIST = [3, 5, 8, 12, 16, 20, 25]
SHIFT_SECONDS = 3600.0
BETWEEN = E.TREE_SPACING/E.TRAVEL

pd.set_option("display.width", 220)
print(f"canopy {len(E.T):,} fruit / {E.T.tree.nunique()} trees")
print(f"arm {PL.HALF_X} x {PL.LIFT} m   pick cycle {E.PICK_SECONDS:.0f} s   "
      f"between trees {BETWEEN:.0f} s")
print(f"k values {K_LIST}   trees {TREES[0]}-{TREES[-1]}")



c:\python\Lib\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


canopy 6,000 fruit / 50 trees
arm 0.2 x 0.6 m   pick cycle 14 s   between trees 3 s
k values [3, 5, 8, 12, 16, 20, 25]   trees 20-49


## 1. Plan each tree at each stop count

Two modes at every count. **Premium** applies a selection threshold and leaves what the outcome
model is unsure about. **Full clear** takes everything the chosen stops reach, which bounds what
the stop count makes available at all.


In [2]:
rows, t0 = [], time.time()
for k in K_LIST:
    for mode, th in (("premium", 0.9), ("full", 0.0)):
        S = PL.plan_summary(TREES, k=k, threshold=th, chooser="greedy", sweep=False)
        S["k"], S["mode"] = k, mode
        rows.append(S)
    print(f"  k={k:<3} {(time.time()-t0)/60:5.1f} min")

SW = pd.concat(rows, ignore_index=True)
SW.to_csv(OUT/"station_sweep.csv", index=False)
print(f"\n{len(SW)} rows -> {OUT/'station_sweep.csv'}")
print(f"columns: {list(SW.columns)}")


  k=3     0.4 min
  k=5     0.7 min
  k=8     1.1 min
  k=12    1.5 min
  k=16    1.9 min
  k=20    2.3 min
  k=25    2.7 min

420 rows -> c:\aipick\git\runs\stations\station_sweep.csv
columns: ['tree', 'stops', 'stage1_stops', 'sweep_stops', 'on_tree', 'robot_reach', 'in_plan', 'attempts', 'seconds', 'travel', 'left_reachable', 'exp_success', 'exp_utility', 'chooser', 'picker', 'k', 'threshold', 'sweep', 'mode']


## 2. What each stop count buys

Per tree, averaged over the thirty held-out trees. `in_plan` is what the chosen stops can touch
at all; `attempts` is what the rule actually tries after the threshold.


In [3]:
for mode in ("full", "premium"):
    M = SW[SW["mode"] == mode]
    T = M.groupby("k").agg(
        stops=("stops", "mean"), in_plan=("in_plan", "mean"),
        attempts=("attempts", "mean"), premium=("exp_success", "mean"),
        utility=("exp_utility", "mean"), seconds=("seconds", "mean"),
        travel=("travel", "mean")).round(2)
    T["cover_pct"] = (T.in_plan/M.robot_reach.mean()*100).round(1)
    T["travel_pct"] = (T.travel/T.seconds.clip(lower=1)*100).round(1)
    T["s_per_premium"] = (T.seconds/T.premium.clip(lower=0.01)).round(1)
    print(f"\n{mode} mode, per tree, {len(TREES)} held-out trees\n")
    print(T[["stops", "in_plan", "cover_pct", "attempts", "premium", "utility",
             "seconds", "travel_pct", "s_per_premium"]].to_string())

print(f"\n  the robot can reach {SW.robot_reach.mean():.1f} fruit with the base free to move")
print("  cover_pct says how much of that the stop count unlocks; s_per_premium is the cost of")
print("  a premium fruit, and if it holds steady as k rises the extra stops are paying for")
print("  themselves even though the tree takes longer.")



full mode, per tree, 30 held-out trees

    stops  in_plan  cover_pct  attempts  premium  utility  seconds  travel_pct  s_per_premium
k                                                                                            
3    3.00    19.57       31.3     19.57    16.69    15.82   351.43        22.1           21.1
5    5.00    29.30       46.8     29.30    24.53    23.11   555.56        26.2           22.6
8    8.00    40.03       64.0     40.03    33.38    31.35   775.55        27.7           23.2
12  12.00    49.77       79.5     49.77    41.44    38.89   977.88        28.8           23.6
16  15.97    55.97       89.5     55.97    46.40    43.43  1101.60        28.9           23.7
20  18.30    59.37       94.9     59.37    48.29    44.94  1172.12        29.1           24.3
25  18.53    59.73       95.5     59.73    48.59    45.22  1178.07        29.0           24.2

premium mode, per tree, 30 held-out trees

    stops  in_plan  cover_pct  attempts  premium  utility  seconds  t

## 3. The two objectives disagree

**Fruit per hour** treats the orchard as unbounded: skim, drive on, and the hour is what is
scarce. **Clearing a block** treats it as finite: every fruit the robot skips is handed to a
person, so the question is how much of the block the machine takes and how long that takes.

The first is what three stops was tuned for. The second is what the client asked for.


In [4]:
N_ORCHARD = 500
DETECTED = 94.3          # per tree, from the canopy; see 06

print("objective A -- premium fruit per hour (unbounded orchard)\n")
print(f"  {'k':>3} {'per tree':>9} {'s/tree':>8} {'trees/h':>9} {'premium/h':>11}")
M = SW[SW["mode"] == "premium"].groupby("k").agg(
    premium=("exp_success", "mean"), seconds=("seconds", "mean"))
best = None
for k, r in M.iterrows():
    per_tree = r.seconds + BETWEEN
    rate = SHIFT_SECONDS/per_tree*r.premium
    if best is None or rate > best[1]:
        best = (k, rate)
    print(f"  {k:>3} {r.premium:>9.1f} {r.seconds:>8.0f} "
          f"{SHIFT_SECONDS/per_tree:>9.1f} {rate:>11.0f}")
rate_all = M.premium/(M.seconds + BETWEEN)*SHIFT_SECONDS
print(f"\n  best at k={best[0]}  ({best[1]:.0f} premium fruit per hour)")
print(f"  across all k the rate spans {rate_all.min():.0f} to {rate_all.max():.0f}, "
      f"a spread of {rate_all.max()/rate_all.min() - 1:.0%}")

print(f"\n\nobjective B -- clearing a block of {N_ORCHARD} trees\n")
print(f"  {'k':>3} {'robot takes':>12} {'% of reach':>11} {'left to people':>15} "
      f"{'robot hours':>12}")
M2 = SW[SW["mode"] == "premium"].groupby("k").agg(
    attempts=("attempts", "mean"), seconds=("seconds", "mean"),
    in_plan=("in_plan", "mean"), robot_reach=("robot_reach", "mean"))
for k, r in M2.iterrows():
    took = r.attempts*N_ORCHARD
    left = (DETECTED - r.attempts)*N_ORCHARD
    hours = (r.seconds + BETWEEN)*N_ORCHARD/3600.0
    print(f"  {k:>3} {took:>12,.0f} {r.in_plan/r.robot_reach*100:>10.0f}% "
          f"{left:>15,.0f} {hours:>12,.0f}")
print("\n  Under B the robot hours rise with k and the human hours fall. Which trade is worth")
print("  taking depends on what a picker costs and on whether the block has to be cleared by a")
print("  date -- both inputs the farm has and this project does not.")



objective A -- premium fruit per hour (unbounded orchard)

    k  per tree   s/tree   trees/h   premium/h
    3      13.8      250      14.2         196
    5      20.0      383       9.3         186
    8      27.1      533       6.7         182
   12      33.0      657       5.5         180
   16      36.1      721       5.0         180
   20      36.6      730       4.9         180
   25      36.8      732       4.9         180

  best at k=3  (196 premium fruit per hour)
  across all k the rate spans 180 to 196, a spread of 9%


objective B -- clearing a block of 500 trees

    k  robot takes  % of reach  left to people  robot hours
    3        7,000         31%          40,150           35
    5       10,183         47%          36,967           54
    8       13,817         64%          33,333           74
   12       16,800         80%          30,350           92
   16       18,400         89%          28,750          101
   20       18,667         95%          28,483         

## 4. Diminishing returns

Where the curve flattens is the practical answer: the stop count past which extra stops buy
little. Read it from the full-clear column, which is what the stops make available before the
threshold takes its share.


In [5]:
F = SW[SW["mode"] == "full"].groupby("k").agg(
    in_plan=("in_plan", "mean"), attempts=("attempts", "mean"),
    seconds=("seconds", "mean")).round(2)
F["d_attempts"] = F.attempts.diff().round(2)
F["d_seconds"] = F.seconds.diff().round(1)
F["s_per_extra"] = (F.d_seconds/F.d_attempts.replace(0, np.nan)).round(1)
print("full clear, marginal cost of raising k\n")
print(F.to_string())
print(f"\n  A pick costs {E.PICK_SECONDS:.0f} s on its own. Where s_per_extra is close to that,")
print(f"  the extra stop is almost free; where it is far above, travel is dominating and the")
print(f"  stop is not worth adding.")

P = SW[SW["mode"] == "premium"].groupby("k").agg(premium=("exp_success", "mean"))
knee = None
ks = list(P.index)
for a, b in zip(ks, ks[1:]):
    if P.loc[b, "premium"] - P.loc[a, "premium"] < 0.5 and knee is None:
        knee = b
print(f"\n  premium gain per step falls below 0.5 fruit at k={knee}" if knee else
      "\n  premium is still rising at the largest k tested -- extend K_LIST")

json.dump(dict(k_list=K_LIST, trees=len(TREES), arm=[PL.HALF_X, PL.LIFT],
               between_trees_s=float(BETWEEN),
               robot_reach=float(SW.robot_reach.mean()),
               best_k_fruit_per_hour=int(best[0]),
               knee_k=(int(knee) if knee else None),
               note="coverage search, no sweep stage, so these are stop-count effects only"),
          open(OUT/"sweep_summary.json", "w"), indent=1)
print(f"\nwritten {OUT/'sweep_summary.json'}")


full clear, marginal cost of raising k

    in_plan  attempts  seconds  d_attempts  d_seconds  s_per_extra
k                                                                 
3     19.57     19.57   351.43         NaN        NaN          NaN
5     29.30     29.30   555.56        9.73      204.1         21.0
8     40.03     40.03   775.55       10.73      220.0         20.5
12    49.77     49.77   977.88        9.74      202.3         20.8
16    55.97     55.97  1101.60        6.20      123.7         20.0
20    59.37     59.37  1172.12        3.40       70.5         20.7
25    59.73     59.73  1178.07        0.36        6.0         16.7

  A pick costs 14 s on its own. Where s_per_extra is close to that,
  the extra stop is almost free; where it is far above, travel is dominating and the
  stop is not worth adding.

  premium gain per step falls below 0.5 fruit at k=25

written c:\aipick\git\runs\stations\sweep_summary.json


### Reading the result

```
premium/h peaks at a small k        confirms three stops was right for that objective,
                                    and wrong for this client -- say both

premium/h is flat across k          the original choice cost nothing; raise k freely

premium/h keeps rising with k       the original tuning was wrong on its own terms,
                                    which is worth reporting plainly
```

Whatever comes out, the number to carry into the report is not "the robot takes N% of a tree" but
the pair: how much of what the arm can reach the stop count unlocks, and what that costs in time.
The first is a design decision the farm can change; the second is what they pay for it.

**The shipping configuration is not on this page.** It uses the trained planner and a sweep
stage, both of which are held off here so that the stop count is the only thing varying. What it
settles on — twenty stops plus a sweep — comes from this curve, and `06_orchard_estimate` reports
what that configuration actually does.
